# K=11 Producao -- Orquestracao (execucao local)

Pipeline de producao do modelo de regressao Bayesiana hierarquica K=11
(10 features baseline + mode_bin) para o produto **Diagnostico de Posicionamento**.

## Pre-requisitos locais

- **Python 3.10+**
- **GPU NVIDIA recomendada** (T4, RTX 3060+, A100). Sem GPU, o NUTS demora ~30h.
- Drivers CUDA 12.x (se for usar GPU)
- ~5 GB de espaco em disco para os artefatos

## Setup rapido

```bash
cd /caminho/para/insights-spotfy-grupo-4
python -m venv .venv
# Windows:
.venv\Scripts\activate
# Linux/Mac:
source .venv/bin/activate

pip install -r requirements.txt

# Se tiver GPU NVIDIA local + CUDA 12 instalado:
pip install --upgrade "jax[cuda12]"

jupyter lab  # ou jupyter notebook
```

Abre este notebook no Jupyter e executa as celulas em ordem.

## Arquitetura

- **Target:** `log(popularity + 1)` -- trata 10% zeros e produz score 0-100 apos `exp() - 1`
- **Features (K=11):** `danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, tempo, explicit, mode_bin`
- **Hierarquica:** 107 generos com intercept e slopes especificos, prior nao-centrado
- **Sampler:** NUTS via NumPyro, 4 chains x 1000 draws, tune=1500, target_accept=0.9
  - **GPU (T4/A100):** ~3h
  - **CPU so:** ~30h (nao recomendado)
- **Validacao:** Train/Val/Test 70/15/15 com SEED=42, asserts RMSE<18, R2>0.30, HDI 0.90-0.97

## Etapas

1. `scripts/train_k11.py` -- fit NUTS (3h em GPU)
2. `scripts/evaluate_k11.py` -- metricas em Val e Test
3. `scripts/export_for_nextjs.py` -- converte NetCDF em JSON para o backend Next.js

In [ ]:
# Instala dependencias (sem [cuda12] por padrao -- o usuario instala manualmente se tiver GPU local)
!pip install -q pymc==6.3.1 arviz==1.3.0 numpyro pandas pyarrow scipy scikit-learn

# jax separado (CPU por default, [cuda12] se tiver GPU NVIDIA + CUDA 12)
try:
    import jax
    print('jax ja instalado:', jax.__version__)
except ImportError:
    print('Instalando jax (CPU)... Para GPU, rode: pip install --upgrade "jax[cuda12]"')
    !pip install -q jax
    import jax

import subprocess
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader,nounits'],
                                  stderr=subprocess.DEVNULL).decode().strip()
    print('GPU detectada:', out)
    print('NUTS deve levar ~3h.')
except Exception:
    print('AVISO: nenhuma GPU NVIDIA detectada (ou nvidia-smi nao esta no PATH).')
    print('NUTS vai rodar em CPU e pode levar ~30h. Considere usar Colab T4 ou uma GPU local.')

In [ ]:
import os
from pathlib import Path

# PROJECT_ROOT = raiz do repo (um nivel acima de notebooks/)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

print('PROJECT_ROOT:', PROJECT_ROOT)
print()
print('Estrutura esperada:')
expected = ['scripts', 'data/processed/spotify_tracks_limpo.parquet', 'artifacts']
for p in expected:
    full = PROJECT_ROOT / p
    status = '[OK]' if full.exists() else '[FALTA]'
    print(f'  {status}  {p}')

print()
print('Scripts:')
for s in sorted((PROJECT_ROOT / 'scripts').glob('*.py')):
    print(f'  {s.name}')

# Assertions para garantir que estamos no lugar certo
assert (PROJECT_ROOT / 'scripts' / 'train_k11.py').exists(), \
    'train_k11.py nao encontrado -- confira se voce abriu o notebook da raiz do repo'
assert (PROJECT_ROOT / 'data' / 'processed' / 'spotify_tracks_limpo.parquet').exists(), \
    'parquet nao encontrado em data/processed/ -- confira o caminho do repo'
print('\nTudo certo. Pode prosseguir.')

In [ ]:
!python scripts/train_k11.py 2>&1 | tee train_k11.log

In [ ]:
!python scripts/evaluate_k11.py 2>&1 | tee evaluate_k11.log

In [ ]:
!python scripts/export_for_nextjs.py 2>&1 | tee export_for_nextjs.log

In [ ]:
import json
from pathlib import Path

print('=== Artefatos gerados ===\n')
for f in sorted(Path('artifacts').iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:45s}  {size_kb:8.1f} KB')

print('\n=== Metricas ===\n')
summary_path = Path('relatorio/analises/resultados/q11_summary.json')
if summary_path.exists():
    with open(summary_path) as fh:
        summary = json.load(fh)
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print('\n[OK] assertions_passed:', summary.get('assertions_passed', False))
else:
    print('AVISO: q11_summary.json nao encontrado -- verifique se evaluate_k11.py rodou sem erro.')

## Proximos passos

### Se `assertions_passed == true:`

1. **Commitar artefatos:**
   ```bash
   git add artifacts/ scripts/ relatorio/analises/resultados/q11_* train_k11.log
   git commit -m "feat: K=11 modelo treinado e validado"
   git push
   ```

2. **Subir o backend Next.js (na mesma maquina ou em outra):**
   ```bash
   cd /caminho/para/insights-spotfy-grupo-4
   cp artifacts/*.json artifacts/*.json.gz ./artifacts/   # garantir artefatos no lugar
   npm install
   cp .env.local.example .env.local
   # editar .env.local e colocar OPENROUTER_API_KEY=sk-or-v1-...
   npm run dev
   ```

3. **Testar o endpoint:**
   ```bash
   curl -X POST http://localhost:3000/api/diagnose \
     -H "Content-Type: application/json" \
     -d '{
       "track_features": {
         "danceability": 0.7, "energy": 0.5, "loudness": -5.0,
         "speechiness": 0.05, "acousticness": 0.3,
         "instrumentalness": 0.0, "liveness": 0.1,
         "valence": 0.6, "tempo": 120.0, "explicit": 0, "mode_bin": 0
       },
       "genero": "sertanejo"
     }'
   ```

### Se `assertions_passed == false:`

Investigar `q11_summary.json` e ver qual metrica falhou:

| Metrica | Falha comum | Acao |
|---------|-------------|------|
| RMSE >= 18 | Modelo nao captura variancia | Aumentar K? (nao recomendado, Q8 v2 mostrou overfit) |
| R2 <= 0.30 | Pouca variancia explicada | Aceitar -- pode ser teto do problema |
| HDI fora de [0.90, 0.97] | Calibracao ruim | Ajustar priors sigma_alpha/sigma_beta |

## Caveats do modelo

- **Genero deve ser conhecido** -- o dropdown tem 107 opcoes apos filtro nao-musical
- **Score e preditivo, nao causal** -- diz "o que costuma acontecer", nao "como fazer hit"
- **Calibrado em popularity do Spotify (0-100)**, nao em qualidade musical
- **NUTS aproxima o posterior** -- HDI e uma estimativa, nao certeza

## Se algo der errado

Logs ficam salvos em:
- `train_k11.log` -- log completo do treino (inclui R-hat, ESS, divergencias)
- `evaluate_k11.log` -- log da avaliacao
- `export_for_nextjs.log` -- log do export

Para debug, rode os scripts diretamente no terminal:
```bash
python scripts/train_k11.py
```
e veja o erro com traceback completo.